## Preparação do Ambiente

In [16]:
# Manipulação de dados
import re
import numpy as np
import pandas as pd

# Visualização
import matplotlib.pyplot as plt

# NLP - NLTK
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer

# Recursos do NLTK
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("rslp", quiet=True)

# Preparação e validação dos dados
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
    learning_curve
)

# Representação textual
from sklearn.feature_extraction.text import TfidfVectorizer

# Modelos de Machine Learning
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

# Pipeline
from sklearn.pipeline import Pipeline

# Avaliação
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay
)

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


## Leitura da Base de Dados

In [3]:
dados = pd.read_csv("../data/banking_train.csv")
dados.head()

,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival


## 1. Pré-Processamento dos Dados

### 1.1 Limpeza e Normalização do Texto

In [4]:
ABBREVIATIONS = {
    "pls": "please", "plz": "please", "thx": "thanks", "idk": "i do not know", "asap": "as soon as possible",
    "rn": "right now", "msg": "message",
    # Conta e transações
    "acct": "account", "accts": "accounts", "txn": "transaction", "txns": "transactions", "transac": "transaction",
    # Valores e informações
    "amt": "amount", "bal": "balance", "avl": "available", "stmt": "statement", "stmts": "statements", "ref": "reference",
    # Cartão e pagamento
    "pymt": "payment", "pymts": "payments", "pmt": "payment", "pmts": "payments", " And": " And",
}

In [5]:
def limpar_texto(texto):

    # 1. Lowercase
    texto = texto.lower()
    # 2. Remover HTML
    texto = re.sub(r'<[^>]+>'," ", texto)
    # 3. Remove URLs
    texto = re.sub(r'http\S+|www\.\S+', " ", texto)
    # 4. Remove e-mails
    texto = re.sub(r'\S+@\S+', " ", texto)
    # 5. Atualizando abreviações
    palavras = [ABBREVIATIONS.get(p,p) for p in texto.split()]
    texto = " ".join(palavras)
    # 6. Remove caracteres indesejados
    texto = re.sub(r'[^a-záàâãéêèíìîóôõöúùûüç\s.,!?]', " ", texto)
    # 7. Normaliza espaços
    texto = re.sub(r'\s+', " ", texto)

    return texto

In [6]:
df_teste = dados.copy()

In [7]:
# Realizando o tratamento como exemplo
df_teste["texto_limpo"] = df_teste["text"].apply(limpar_texto)

df_teste[["text", "texto_limpo"]].head()

,text,texto_limpo
0,I am still waiting on my card?,i am still waiting on my card?
1,What can I do if my card still hasn't arrived ...,what can i do if my card still hasn t arrived ...
2,I have been waiting over a week. Is the card s...,i have been waiting over a week. is the card s...
3,Can I track my card while it is in the process...,can i track my card while it is in the process...
4,"How do I know if I will get my card, or if it ...","how do i know if i will get my card, or if it ..."


### 1.2 Tokenização

In [8]:
def tokenizacao(texto):

    return word_tokenize(texto, language="english")

In [9]:
# Aplicando a tokenização dos dados em teste
df_teste["texto_tokenizado"] = df_teste["texto_limpo"].apply(tokenizacao)

df_teste[["texto_limpo", "texto_tokenizado"]].head()

,texto_limpo,texto_tokenizado
0,i am still waiting on my card?,"[i, am, still, waiting, on, my, card, ?]"
1,what can i do if my card still hasn t arrived ...,"[what, can, i, do, if, my, card, still, hasn, ..."
2,i have been waiting over a week. is the card s...,"[i, have, been, waiting, over, a, week, ., is,..."
3,can i track my card while it is in the process...,"[can, i, track, my, card, while, it, is, in, t..."
4,"how do i know if i will get my card, or if it ...","[how, do, i, know, if, i, will, get, my, card,..."


### 1.3 Remoção de StopWords (tirando a parte de negação)

In [10]:
STOP_WORDS = set(stopwords.words("english"))

NEGATIONS = {
    "no", "not", "nor", "never", "neither",
    "none", "nobody", "nothing", "nowhere",
    "cannot", "without",

    # Contrações negativas
    "don't", "doesn't", "didn't",
    "isn't", "aren't", "wasn't", "weren't",
    "haven't", "hasn't", "hadn't",
    "won't", "wouldn't", "can't", "couldn't",
    "shouldn't", "mustn't", "needn't",
}

STOP_WORDS_EN = STOP_WORDS - NEGATIONS

In [11]:
def remove_stopwords(tokens):
    resultado = []
    for token in tokens:
        if token.isalpha() and token not in STOP_WORDS_EN:
            resultado.append(token)
    
    return resultado

In [12]:
df_teste["texto_stopwords"] = df_teste["texto_tokenizado"].apply(remove_stopwords)

df_teste[["texto_tokenizado", "texto_stopwords"]].head()

,texto_tokenizado,texto_stopwords
0,"[i, am, still, waiting, on, my, card, ?]","[still, waiting, card]"
1,"[what, can, i, do, if, my, card, still, hasn, ...","[card, still, arrived, weeks]"
2,"[i, have, been, waiting, over, a, week, ., is,...","[waiting, week, card, still, coming]"
3,"[can, i, track, my, card, while, it, is, in, t...","[track, card, process, delivery]"
4,"[how, do, i, know, if, i, will, get, my, card,...","[know, get, card, lost]"


### 1.4 Stemming

In [17]:
stemmer = SnowballStemmer("english")

def aplicar_stemming(tokens):
    resultado = []
    for token in tokens:
        resultado.append(stemmer.stem(token))
    
    return resultado

In [18]:
df_teste["texto_stemmer"] = df_teste["texto_stopwords"].apply(aplicar_stemming)

df_teste[["texto_stopwords", "texto_stemmer"]].head()

,texto_stopwords,texto_stemmer
0,"[still, waiting, card]","[still, wait, card]"
1,"[card, still, arrived, weeks]","[card, still, arriv, week]"
2,"[waiting, week, card, still, coming]","[wait, week, card, still, come]"
3,"[track, card, process, delivery]","[track, card, process, deliveri]"
4,"[know, get, card, lost]","[know, get, card, lost]"


### 1.5 Pipeline Completo

In [19]:
def pipeline_completo(texto):
    texto = limpar_texto(texto)
    tokens = tokenizacao(texto)
    texto_sem_stop_words = remove_stopwords(tokens)
    texto_stemmer = aplicar_stemming(texto_sem_stop_words)

    return " ".join(texto_stemmer)

In [20]:
dados["texto_tratado"] = dados["text"].apply(pipeline_completo)

dados[["text", "texto_tratado"]].head()

,text,texto_tratado
0,I am still waiting on my card?,still wait card
1,What can I do if my card still hasn't arrived ...,card still arriv week
2,I have been waiting over a week. Is the card s...,wait week card still come
3,Can I track my card while it is in the process...,track card process deliveri
4,"How do I know if I will get my card, or if it ...",know get card lost


# 2. Preparação dos Dados

### 2.1 Divisão dos Dados

In [21]:
dados.columns

Index(['text', 'category', 'texto_tratado'], dtype='str')

In [22]:
X = dados["texto_tratado"]
y = dados["category"]

In [23]:
X_treino, X_val, y_treino, y_val = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

print(f"Dados de Treino -> {len(X_treino)}")
print(f"Dados de Validação -> {len(X_val)}")

Dados de Treino -> 8002
Dados de Validação -> 2001


### 2.2 Vetorização com TF-IDF

In [25]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1,2),
    min_df=5,
    max_df=0.85,
    max_features=10000,
    sublinear_tf=True
)

In [26]:
X_tfidf = vectorizer.fit_transform(X_treino)
print(X_tfidf.shape)

(8002, 1926)


In [27]:
features_names = vectorizer.get_feature_names_out()

print(f"Quantidade de Features: {len(features_names)}")
print(features_names[:50])

Quantidade de Features: 1926
['abl' 'abl get' 'abl see' 'abl top' 'abl use' 'abl withdraw' 'abroad'
 'abroad exchang' 'accept' 'accept card' 'accept mastercard' 'access'
 'access account' 'access app' 'access card' 'accident' 'accord' 'account'
 'account alreadi' 'account anoth' 'account app' 'account balanc'
 'account bank' 'account charg' 'account check' 'account children'
 'account daughter' 'account detail' 'account even' 'account get'
 'account happen' 'account help' 'account inform' 'account make'
 'account need' 'account not' 'account number' 'account one'
 'account pleas' 'account possibl' 'account rememb' 'account say'
 'account see' 'account servic' 'account show' 'account still'
 'account transfer' 'account use' 'account via' 'account work']


In [29]:
doc = X_tfidf[0].toarray().flatten()

top_idx = np.argsort(doc)[-10:][::-1]

for i in top_idx:
    if doc[i] > 0:
        print(f"{features_names[i]:20} {doc[i]:.3f}")

take show            0.507
transfer take        0.400
long transfer        0.396
show account         0.388
long                 0.265
take                 0.260
show                 0.257
transfer             0.189
account              0.187


# 3. Modelos de NLP

### 3.1 Naive Bayes

In [30]:
pipeline_nb = Pipeline([
    ('tfidf', vectorizer),
    ('modelo_nb', MultinomialNB(alpha=1.0))
])

pipeline_nb.fit(X_treino, y_treino)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('modelo_nb', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[<U48](77,)","['Refund_not_showing_up','activate_my_card','age_limit',..., 'why_verify_identity','wrong_amount_of_cash_received', 'wrong_exchange_rate_for_cash_withdrawal']"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"max_df max_df: float or int, default=1.0When building the vocabulary ignore terms that have a documentfrequency strictly higher than the given threshold (corpus-specificstop words).If float in range [0.0, 1.0], the parameter represents a proportion ofdocuments, integer absolute counts.This parameter is ignored if vocabulary is not None.",0.85
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored if vocabulary is not None.",5
,"max_features max_features: int, default=NoneIf not None, build a vocabulary that only consider the top`max_features` ordered by term frequency across the corpus.Otherwise, all features are used.This parameter is ignored if vocabulary is not None.",10000
,"sublinear_tf sublinear_tf: bool, default=FalseApply sublinear tf scaling, i.e. replace tf with 1 + log(tf).",True


In [33]:
previsao_nb = pipeline_nb.predict(X_val)

print("=== Resultados Modelo Naive Bayes ===")
print(classification_report(y_val, previsao_nb))

=== Resultados Modelo Naive Bayes ===
                                                  precision    recall  f1-score   support

                           Refund_not_showing_up       0.94      0.91      0.92        32
                                activate_my_card       0.75      0.94      0.83        32
                                       age_limit       1.00      0.91      0.95        22
                         apple_pay_or_google_pay       0.96      1.00      0.98        25
                                     atm_support       0.88      0.41      0.56        17
                                automatic_top_up       1.00      0.96      0.98        25
         balance_not_updated_after_bank_transfer       0.70      0.62      0.66        34
balance_not_updated_after_cheque_or_cash_deposit       0.74      0.97      0.84        36
                         beneficiary_not_allowed       0.76      0.81      0.78        31
                                 cancel_transfer       0.79  

c:\Users\Unisalesiano\Documents\Renan\Projetos Renan\banking-intent-classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Unisalesiano\Documents\Renan\Projetos Renan\banking-intent-classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Unisalesiano\Documents\Renan\Projetos Renan\banking-intent-classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in lab